# Doping and Gap at the Dirac Point of Graphene on SiO2

> **Yong-Ju Kang, Joongoo Kang, and K. J. Chang**, "Electronic structure of graphene and doping
> effect on SiO2", Physical Review B 78, 115404 (2008).
> [DOI:10.1103/PhysRevB.78.115404](https://doi.org/10.1103/PhysRevB.78.115404)

Computes the band structure of the graphene on O-terminated α-quartz(0001) interface created in the
[structure notebook](interface_2d_3d_graphene_silicon_dioxide.ipynb), then reads the Dirac point's
position relative to the Fermi level and the gap at K, and compares both with Kang et al. (2008)
Sec. III and Fig. 3(a).

<h2 style="color:green">Usage</h2>

1. Create the interface in the [structure notebook](interface_2d_3d_graphene_silicon_dioxide.ipynb), which saves it to the `uploads` folder under the name used in cell 1.2 below.
1. Set the material name and parameters in cells 1.2-1.4, or use the defaults; `RELAX = True` relaxes the interface once (fixed-cell, whole slab) and saves it as `<name> relaxed` for reuse.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the results; the last section prints the comparison with the manuscript.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material name, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the interface by name from the `uploads` folder, print its provenance and save it to the platform.
1. Configure the DFT model and k-grid.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Relax the interface if `RELAX`, reusing a saved relaxed structure when one already exists.
1. Configure the Band Structure workflow and run the job, re-using a job that already ran under the same workflow name.
1. Retrieve the band structure, the Fermi level and the Dirac point's position and gap at K.
1. Compare with Kang et al. (2008).


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")

### 1.2. Material name

In [ ]:
# Name saved by interface_2d_3d_graphene_silicon_dioxide.ipynb.
INTERFACE_NAME = "C(001)-O2Si(001), Interface, Strain 1.875pct"

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
BAND_STRUCTURE_WORKFLOW_SEARCH_TERM = "band_structure.json"
MY_WORKFLOW_NAME = "Band Structure"
APPLICATION_NAME = "espresso"

# NOTE: False reads the band structure as built; True relaxes the interface once (fixed cell, whole
# slab) and reads the band structure off the relaxed structure.
RELAX = False

CLUSTER_NAME = "001"  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OR
PPN = 16  # queue OR on cluster-001 allows at most 16 cores per node
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
MODEL_SUBTYPE = "lda"
FUNCTIONAL = "pz"  # Kang et al. 2008 use LDA
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the only LDA family the platform publishes for Si, O and C
GBRV_VALENCE = {"Si": 4, "O": 6, "C": 4}  # valence electrons per atom of the GBRV pseudopotentials
ECUTWFC = 40   # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff

KPOINT_DENSITY = 4  # gives 6 x 6 x 1 on the 1x1 quartz cell, Kang et al. Sec. II
SMEARING_SETTINGS = {"degauss": 0.01}  # Ry; the doped graphene has no gap at E_F
KPATH_STEPS = 20
MODEL_TAG = (f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KPOINT_DENSITY} "
             f"p{KPATH_STEPS} g{SMEARING_SETTINGS['degauss']}")

SCF_UNIT = "pw_scf"
BANDS_UNIT = "pw_bands"
RELAX_UNIT = "pw_relax"
RELAXATION_SETTINGS = {"forc_conv_thr": 1.17e-3, "nstep": 100}  # 0.03 eV/Å, Kang et al. 2008 Sec. II
# Names the relaxation job; the relaxed structure itself is found by content hash.
RELAX_TAG = f"{MODEL_TAG} f{RELAXATION_SETTINGS['forc_conv_thr']}"

KPATH = [
    {"point": "K", "steps": KPATH_STEPS},
    {"point": "Γ", "steps": KPATH_STEPS},
    {"point": "M", "steps": KPATH_STEPS},
    {"point": "K", "steps": 1},
]
K_INDEX = 0  # KPATH starts at K, so the first point of the band structure's path is K

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the material
### 3.1. Load from the uploads folder and print provenance

The structure, its registry and its 120° hexagonal cell come from the structure notebook; this one
only loads it by name.

In [ ]:
import numpy as np
from collections import Counter
from mat3ra.notebooks_utils.core.entity.material.api import load_material

interface = load_material(client, FOLDER, INTERFACE_NAME, ACCOUNT_ID)
interface.basis.set_labels_from_list([])

composition = Counter(interface.basis.elements.values)
electron_count = sum(GBRV_VALENCE[element] * count for element, count in composition.items())
number_of_occupied_bands = electron_count // 2

interface_in_cartesian = interface.clone()
interface_in_cartesian.to_cartesian()
z_coordinates = np.array(interface_in_cartesian.basis.coordinates.values)[:, 2]
is_carbon = np.array(interface.basis.elements.values) == "C"
interlayer_distance = z_coordinates[is_carbon].min() - z_coordinates[~is_carbon].max()
print(f"{INTERFACE_NAME}: {dict(composition)}, {interface.basis.number_of_atoms} atoms, "
      f"gamma = {interface.lattice.gamma:.3f}°, interlayer distance = {interlayer_distance:.3f} Å, "
      f"{electron_count} valence electrons, {number_of_occupied_bands} occupied bands")

### 3.2. Preview the material

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

visualize_materials([{"material": interface, "title": INTERFACE_NAME}], viewer="wave")

### 3.3. Save the material to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_interface = get_or_create_material(client, interface, ACCOUNT_ID)

## 4. Configure the DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

### 4.2. k-grid

In [ ]:
from mat3ra.notebooks_utils.workflow import kgrid_from_density

kgrid = kgrid_from_density(interface, KPOINT_DENSITY, periodic_dims=(0, 1))
print(f"k-grid {kgrid}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Relax the interface (optional)

Runs only if `RELAX`: finds any relaxed version of this structure already on the account,
regardless of who relaxed it or with what, before spending any compute on a new one. The band
structure is then read off that geometry.

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.core.entity.material.api import find_relaxed_material, get_final_structure_for_job
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

relaxed_interface = None
if RELAX:
    relax_workflow_name = f"Fixed-cell Relaxation {INTERFACE_NAME} {RELAX_TAG}"
    relaxed_interface = find_relaxed_material(client, interface, ACCOUNT_ID)
    if relaxed_interface is not None:
        print(f"♻️  Relaxed interface: {relaxed_interface.name} ({relaxed_interface.id})")
    else:
        relax_workflow = Workflow.create(
            WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(RELAX_WORKFLOW_SEARCH_TERM)
        )
        relax_workflow.name = relax_workflow_name
        relax_workflow.subworkflows[0].model = model
        apply_planewave_cutoffs(relax_workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
        apply_scf_kgrid(relax_workflow, kgrid, material=interface, unit_name=RELAX_UNIT)
        patch_workflow_qe_input(relax_workflow, {"system": SMEARING_SETTINGS}, [RELAX_UNIT])
        patch_workflow_qe_input(relax_workflow, {"control": RELAXATION_SETTINGS}, [RELAX_UNIT])

        relax_job = find_job_for_material(
            client, saved_interface["_id"], relax_workflow_name, ACCOUNT_ID,
            statuses=("submitted", "queued", "active", "finished"),
        )
        if relax_job is None:
            relax_job = create_job(
                api_client=client, materials=[saved_interface], workflow=relax_workflow,
                project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
                prefix=f"{relax_workflow_name} {timestamp}",
            )
            submit_jobs(client.jobs, [relax_job["_id"]])
        await wait_for_jobs_to_finish_async(client.jobs, [relax_job["_id"]], poll_interval=POLL_INTERVAL)

        relaxed_material = get_final_structure_for_job(client, relax_job["_id"])
        client.materials.update(relaxed_material.id, {"name": f"{INTERFACE_NAME} relaxed"})
        relaxed_interface = Material.create(client.materials.get(relaxed_material.id))
        print(f"✅ Relaxed interface: {relaxed_interface.name} ({relaxed_interface.id})")

        total_force = get_properties_for_job(client, relax_job["_id"], "total_force")[0]
        print(f"Residual force after relaxation (norm over all atoms): "
              f"{total_force['value']:.4f} {total_force['units']}")

interface_for_job = relaxed_interface if RELAX else interface
interface_for_job.lattice.type = "HEX"  # the symbolic K-Γ-M-K path needs a named lattice type
saved_interface_for_job = get_or_create_material(client, interface_for_job, ACCOUNT_ID)

## 7. Band structure job
### 7.1. Configure the workflow

In [ ]:
from mat3ra.wode.context.providers import PointsPathDataProvider
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow = Workflow.create(
    WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(BAND_STRUCTURE_WORKFLOW_SEARCH_TERM)
)
workflow.name = f"{MY_WORKFLOW_NAME} {INTERFACE_NAME} {MODEL_TAG}"
workflow.subworkflows[0].model = model
apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=SCF_UNIT)
apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=BANDS_UNIT)
apply_scf_kgrid(workflow, kgrid, material=interface_for_job)
patch_workflow_qe_input(workflow, {"system": SMEARING_SETTINGS}, [SCF_UNIT, BANDS_UNIT])

bands_subworkflow = workflow.subworkflows[0]
bands_unit = bands_subworkflow.get_unit_by_name(name=BANDS_UNIT)
bands_unit.add_context(PointsPathDataProvider(path=KPATH, isEdited=True).get_context_item_data())
bands_subworkflow.set_unit(bands_unit)

visualize_workflow(workflow)

### 7.2. Run the job

A job is found by its material and its workflow name, which carries the model tag, and re-used if it exists; otherwise it is created and submitted.

In [ ]:
job = find_job_for_material(
    client, saved_interface_for_job["_id"], workflow.name, ACCOUNT_ID,
    statuses=("submitted", "queued", "active", "finished"),
)
if job is None:
    job = create_job(
        api_client=client, materials=[saved_interface_for_job], workflow=workflow, project_id=project_id,
        owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
    )
    submit_jobs(client.jobs, [job["_id"]])
    print(f"✅ Submitted job {job['_id']}")
else:
    print(f"♻️  Reusing job {job['_id']}")
await wait_for_jobs_to_finish_async(client.jobs, [job["_id"]], poll_interval=POLL_INTERVAL)
job_id = job["_id"]

## 8. Retrieve the results
### 8.1. Band structure

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

band_structure = get_properties_for_job(client, job_id, property_name="band_structure")
visualize_properties(band_structure, title="Band Structure",
                     extra_config={"material": interface_for_job.to_dict()})

### 8.2. Dirac point and gap at K

With `number_of_occupied_bands` doubly occupied bands, the two Dirac bands at K are bands
`number_of_occupied_bands - 1` and `number_of_occupied_bands` (0-based) for neutral graphene. If a
surface O state sits below the Dirac point, as in p-doped graphene, they are `number_of_occupied_bands`
and `number_of_occupied_bands + 1`. The pair with the smaller splitting at K is the Dirac pair; the
Dirac point is its midpoint and the gap its splitting.

In [ ]:
bands = np.array(band_structure[0]["yDataSeries"])
fermi_energy = band_structure[0]["fermiEnergy"]
energies_at_k = bands[:, K_INDEX]
print(f"Bands at K relative to E_F (eV), 0-based indices {number_of_occupied_bands - 2}..{number_of_occupied_bands + 2}:")
for index in range(number_of_occupied_bands - 2, number_of_occupied_bands + 3):
    print(f"  {index}: {energies_at_k[index] - fermi_energy:+.4f}")

candidates = [(number_of_occupied_bands - 1, number_of_occupied_bands), (number_of_occupied_bands, number_of_occupied_bands + 1)]
below, above = min(candidates, key=lambda pair: energies_at_k[pair[1]] - energies_at_k[pair[0]])
gap = energies_at_k[above] - energies_at_k[below]
dirac_minus_fermi = (energies_at_k[above] + energies_at_k[below]) / 2 - fermi_energy
print(f"Dirac pair: bands {below} and {above}")
print(f"E_F = {fermi_energy:.4f} eV, E_D - E_F = {dirac_minus_fermi:+.4f} eV, gap at K = {1000 * gap:.1f} meV")

## 9. Compare with Kang et al. (2008)

Settings that differ from the manuscript: GBRV ultrasoft LDA pseudopotentials against VASP at 396 eV;
the standata quartz cell is +2.3 % on a against the manuscript's 4.91 Å; the substrate is 5
conventional quartz cells (15 Si planes) where the manuscript has 14 bilayers; the back surface is the bare cut, not
H-passivated.

In [ ]:
KANG_DIRAC_MINUS_FERMI = 1.2  # eV, Fig. 3(a) read-off
KANG_GAP = 0.13  # eV, Sec. III text

print(f"Regime: {'relaxed interface' if RELAX else 'unrelaxed SCF'}")
print(f"Doping: {'p-type' if dirac_minus_fermi > 0 else 'n-type'} (Kang et al., 2008: p-type)")
print(f"{'E_D - E_F (Kang et al., 2008):':<32}{KANG_DIRAC_MINUS_FERMI:+.3f} eV")
print(f"{'E_D - E_F (this notebook):':<32}{dirac_minus_fermi:+.3f} eV "
      f"({100 * (dirac_minus_fermi - KANG_DIRAC_MINUS_FERMI) / KANG_DIRAC_MINUS_FERMI:+.1f} % deviation)")
print(f"{'Gap at K (Kang et al., 2008):':<32}{KANG_GAP:.3f} eV")
print(f"{'Gap at K (this notebook):':<32}{gap:.3f} eV "
      f"({100 * (gap - KANG_GAP) / KANG_GAP:+.1f} % deviation)")

## References

[1] Y.-J. Kang, J. Kang and K. J. Chang, "Electronic structure of graphene and doping effect on SiO2", Phys. Rev. B 78, 115404 (2008). https://doi.org/10.1103/PhysRevB.78.115404

[2] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053